# F03 Sweeps, Ablations, and Failure Analysis


## What this foundation lab is for

This lab exists to reduce the most common beginner failure modes before the article-first path starts.


## Skills you should leave with

- Design the smallest useful sweep around one variable family.
- Use ablations to rule out explanations instead of merely deleting a component.
- Write down a stop condition, a failure mode, and the next step.


## Ship these outputs

- One sweep table.
- One ablation comparison plot.
- One failure-analysis memo.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(19)
train_x = torch.randn(320, 3)
train_signal = 1.3 * train_x[:, 0] - 0.9 * train_x[:, 1]
train_x[:, 2] = 0.8 * (train_signal > 0).float() + 0.2 * torch.randn(320)
train_y = (train_signal > 0).float().unsqueeze(1)

ood_x = torch.randn(160, 3)
ood_signal = 1.3 * ood_x[:, 0] - 0.9 * ood_x[:, 1]
ood_x[:, 2] = 0.8 * (ood_signal < 0).float() + 0.2 * torch.randn(160)
ood_y = (ood_signal > 0).float().unsqueeze(1)


def train_model(weight_decay: float):
    torch.manual_seed(31)
    model = nn.Linear(3, 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.08, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(120):
        optimizer.zero_grad()
        loss = loss_fn(model(train_x), train_y)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        train_acc = float(((torch.sigmoid(model(train_x)) > 0.5).float().eq(train_y)).float().mean())
        ood_acc = float(((torch.sigmoid(model(ood_x)) > 0.5).float().eq(ood_y)).float().mean())
        weights = model.weight.detach().clone().squeeze(0)
        bias = float(model.bias.detach().clone().squeeze(0))
    return train_acc, ood_acc, weights, bias


def evaluate_ablation(weights: torch.Tensor, bias: float):
    ablated_x = ood_x.clone()
    ablated_x[:, 2] = 0.0
    logits = ablated_x @ weights[:3] + bias
    preds = (torch.sigmoid(logits.unsqueeze(1)) > 0.5).float()
    return float(preds.eq(ood_y).float().mean())


weight_decays = [0.0, 0.001, 0.01, 0.05, 0.1]
records = []
for wd in weight_decays:
    train_acc, ood_acc, weights, bias = train_model(wd)
    ablated_acc = evaluate_ablation(weights, bias)
    records.append((wd, train_acc, ood_acc, ablated_acc, float(weights[2])))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
positions = list(range(len(weight_decays)))
axes[0].plot(positions, [row[1] for row in records], marker="o", label="train")
axes[0].plot(positions, [row[2] for row in records], marker="o", label="OOD")
axes[0].plot(positions, [row[3] for row in records], marker="o", label="OOD after ablation")
axes[0].set_title("Sweep over weight decay")
axes[0].set_xlabel("weight decay")
axes[0].set_xticks(positions, [str(value) for value in weight_decays], rotation=25)
axes[0].legend()

axes[1].bar([str(row[0]) for row in records], [row[4] for row in records], color="#c96a28")
axes[1].set_title("Weight placed on the spurious feature")
axes[1].set_xlabel("weight decay")
plt.tight_layout()

for wd, train_acc, ood_acc, ablated_acc, spurious_weight in records:
    print(
        f"wd={wd:>5} | train={train_acc:.3f} | OOD={ood_acc:.3f} | "
        f"OOD after ablation={ablated_acc:.3f} | spurious weight={spurious_weight:.3f}"
    )


## Takeaway

The point of a sweep is not to produce many numbers. The point is to locate where your interpretation breaks or becomes robust.


## Self-check questions

- What kind of sweep is merely 'running a few more times' rather than actual experiment design?
- When an ablation does not change the result, how do you tell a negative result from a bad experiment?
- If the signal is weak, how do you decide to stop instead of appending more trials forever?
- If you cannot answer at least two questions without reopening the notebook, stay here before moving to the article track.
